# Fake vs True News VAD Analysis

This notebook compares valence, arousal, and dominance (VAD) patterns between fake and true news articles in `data/FakeVsTrueVAD`.

Flow:
1) loads the dataset through `src`
2) loads a pretrained VAD model from Hugging Face
3) scores each article with the model
4) compares `fake` vs `true` summaries and subject-level patterns
5) exports row-level and summary CSVs

Default model: `RobroKools/vad-bert`.
Note: the first run may download the model weights from Hugging Face.

## 1) Imports

In [ ]:
from pathlib import Path

from IPython.display import display

from misinformation_simulation.audits import (
    DEFAULT_VAD_MODEL_NAME,
    annotate_vad_scores,
    compare_vad_groups,
    load_huggingface_vad_model,
    summarize_vad_by_group,
)
from misinformation_simulation.datasets import load_fake_true_news_dataset

## 2) Configuration

In [ ]:
DATA_DIR = Path("../data/FakeVsTrueVAD")
OUTPUT_DIR = Path("../output/audit/VADPatternAudit")
TEXT_COLUMN = "article_text"
MODEL_NAME = DEFAULT_VAD_MODEL_NAME
MAX_ROWS_PER_LABEL = 10000  # Balanced sample: 1000 fake + 1000 true.
BATCH_SIZE = 16
SHOW_PROGRESS = True
SUBJECT_TOP_N = 10

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"DATA_DIR: {DATA_DIR.resolve()}")
print(f"OUTPUT_DIR: {OUTPUT_DIR.resolve()}")
print(f"MODEL_NAME: {MODEL_NAME}")
print(f"MAX_ROWS_PER_LABEL: {MAX_ROWS_PER_LABEL}")
print(f"BATCH_SIZE: {BATCH_SIZE}")
print(f"SHOW_PROGRESS: {SHOW_PROGRESS}")

## 3) Load the dataset

In [ ]:
news_df = load_fake_true_news_dataset(
    DATA_DIR,
    max_rows_per_label=MAX_ROWS_PER_LABEL,
    output_text_column=TEXT_COLUMN,
)

dataset_overview = (
    news_df.groupby("label")
    .agg(
        articles=("label", "size"),
        avg_chars=("article_char_count", "mean"),
        avg_words=("article_word_count", "mean"),
    )
    .reset_index()
)

display(dataset_overview)
news_df[["label", "subject", "title", TEXT_COLUMN]].head(3)

## 4) Load the pretrained VAD model

In [ ]:
vad_model = load_huggingface_vad_model(model_name=MODEL_NAME)
print(f"Loaded model: {vad_model.model_name}")
print(f"Device: {vad_model.device}")
print(f"Max length: {vad_model.max_length}")

## 5) Score each article with the VAD model

In [ ]:
vad_df = annotate_vad_scores(
    news_df,
    text_column=TEXT_COLUMN,
    model_bundle=vad_model,
    batch_size=BATCH_SIZE,
    show_progress=SHOW_PROGRESS,
    progress_description="VAD progress",
)

row_level_output = OUTPUT_DIR / "fake_true_vad_scored.csv"
vad_df.to_csv(row_level_output, index=False)

display(
    vad_df[
        [
            "label",
            "title",
            "vad_valence",
            "vad_arousal",
            "vad_dominance",
        ]
    ].head(10)
)
print(f"Row-level scores exported to: {row_level_output.resolve()}")

## 6) Global comparison: fake vs true

In [ ]:
label_summary_df = summarize_vad_by_group(vad_df, group_column="label")
comparison_df = compare_vad_groups(vad_df)

label_summary_output = OUTPUT_DIR / "fake_true_vad_label_summary.csv"
comparison_output = OUTPUT_DIR / "fake_true_vad_group_comparison.csv"
label_summary_df.to_csv(label_summary_output, index=False)
comparison_df.to_csv(comparison_output, index=False)

display(label_summary_df)
display(comparison_df)

print(f"Label summary exported to: {label_summary_output.resolve()}")
print(f"Group comparison exported to: {comparison_output.resolve()}")

## 7) Subject-level patterns

In [ ]:
subject_summary_df = (
    vad_df.groupby(["label", "subject"], dropna=False)
    .agg(
        article_count=("label", "size"),
        valence_mean=("vad_valence", "mean"),
        arousal_mean=("vad_arousal", "mean"),
        dominance_mean=("vad_dominance", "mean"),
    )
    .reset_index()
    .sort_values(["label", "article_count"], ascending=[True, False])
)

subject_summary_output = OUTPUT_DIR / "fake_true_vad_subject_summary.csv"
subject_summary_df.to_csv(subject_summary_output, index=False)

top_subjects_df = subject_summary_df.groupby("label", group_keys=False).head(SUBJECT_TOP_N)
display(top_subjects_df)
print(f"Subject summary exported to: {subject_summary_output.resolve()}")

## 8) Examples and quick interpretation

In [ ]:
columns_to_show = [
    "label",
    "subject",
    "title",
    "vad_valence",
    "vad_arousal",
    "vad_dominance",
]

high_arousal_examples = vad_df.sort_values("vad_arousal", ascending=False)[columns_to_show].head(10)
low_valence_examples = vad_df.sort_values("vad_valence", ascending=True)[columns_to_show].head(10)
high_dominance_examples = vad_df.sort_values("vad_dominance", ascending=False)[
    columns_to_show
].head(10)

display(high_arousal_examples)
display(low_valence_examples)
display(high_dominance_examples)

comparison_lookup = comparison_df.set_index("metric")
print("Quick interpretation:")
print(comparison_lookup[["baseline_mean", "comparison_mean", "mean_diff", "mannwhitney_pvalue"]])
print(f"All artifacts saved to: {OUTPUT_DIR.resolve()}")